<a href="https://colab.research.google.com/github/dorhoffman/SWINGPULSE/blob/main/notebooks/09_Final_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import yfinance as yf

PROJECT_PATH = "/content/drive/MyDrive/SWINGPULSE"
MODELS_FOLDER = f"{PROJECT_PATH}/models"
UTILS_FOLDER = f"{PROJECT_PATH}/utils"

MODEL_PATH = f"{MODELS_FOLDER}/random_forest_model.joblib"
METADATA_PATH = f"{MODELS_FOLDER}/model_metadata.json"

if UTILS_FOLDER not in sys.path:
    sys.path.append(UTILS_FOLDER)

from feature_engineering import add_technical_features

model = joblib.load(MODEL_PATH)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:
    metadata = json.load(file)

FEATURE_COLUMNS = metadata["features"]
DECISION_THRESHOLD = metadata["decision_threshold"]
PREDICTION_HORIZON = metadata["prediction_horizon"]
TARGET_RETURN = metadata["target_return"]

print("SWINGPULSE loaded successfully")
print("Model:", metadata["selected_model"])
print("Decision threshold:", DECISION_THRESHOLD)

Mounted at /content/drive
SWINGPULSE loaded successfully
Model: Random Forest
Decision threshold: 0.45


In [2]:
def download_live_stock(
    symbol: str,
    period: str = "2y"
) -> pd.DataFrame:

    symbol = str(symbol).upper().strip()

    data = yf.download(
        symbol,
        period=period,
        interval="1d",
        auto_adjust=False,
        actions=True,
        progress=False,
        threads=False
    )

    if data.empty:
        raise ValueError(
            f"No Yahoo Finance data returned for {symbol}."
        )

    data = data.reset_index()

    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data.columns = [
        str(column).strip().replace(" ", "_")
        for column in data.columns
    ]

    if "Adj_Close" in data.columns:
        data = data.drop(columns=["Adj_Close"])

    data["Symbol"] = symbol

    return data

In [3]:
def analyze_live_stock(symbol: str) -> dict:
    symbol = str(symbol).upper().strip()

    try:
        live_data = download_live_stock(symbol)

        live_features = add_technical_features(
            live_data
        )

        valid_rows = live_features.dropna(
            subset=FEATURE_COLUMNS
        )

        if valid_rows.empty:
            return {
                "success": False,
                "message": (
                    f"Not enough historical data for {symbol}."
                )
            }

        latest_row = valid_rows.iloc[-1]

        model_input = pd.DataFrame(
            [latest_row[FEATURE_COLUMNS].to_dict()],
            columns=FEATURE_COLUMNS
        ).replace([np.inf, -np.inf], np.nan)

        probability = model.predict_proba(
            model_input
        )[0, 1]

        if probability >= DECISION_THRESHOLD + 0.15:
            signal = "Strong Watch"
        elif probability >= DECISION_THRESHOLD:
            signal = "Watch"
        elif probability >= DECISION_THRESHOLD - 0.10:
            signal = "Neutral"
        else:
            signal = "Low Potential"

        return {
            "success": True,
            "symbol": symbol,
            "date": latest_row["Date"],
            "close": float(latest_row["Close"]),
            "probability": float(probability),
            "signal": signal,
            "rsi": float(latest_row["RSI_14"]),
            "macd": float(latest_row["MACD"]),
            "macd_signal": float(
                latest_row["MACD_Signal"]
            ),
            "ema_20": float(latest_row["EMA_20"]),
            "ema_50": float(latest_row["EMA_50"]),
            "ema_200": float(latest_row["EMA_200"]),
            "atr": float(latest_row["ATR_14"]),
            "volatility": float(
                latest_row["Volatility_20"]
            )
        }

    except Exception as error:
        return {
            "success": False,
            "message": str(error)
        }

In [4]:
def display_live_analysis(symbol: str):
    result = analyze_live_stock(symbol)

    if not result["success"]:
        print(result["message"])
        return

    if result["rsi"] >= 70:
        rsi_status = "Overbought"
    elif result["rsi"] <= 30:
        rsi_status = "Oversold"
    elif result["rsi"] >= 55:
        rsi_status = "Positive momentum"
    elif result["rsi"] <= 45:
        rsi_status = "Weak momentum"
    else:
        rsi_status = "Neutral momentum"

    macd_status = (
        "Bullish"
        if result["macd"] > result["macd_signal"]
        else "Bearish"
    )

    if (
        result["ema_20"]
        > result["ema_50"]
        > result["ema_200"]
    ):
        trend = "Strong bullish trend"

    elif (
        result["ema_20"]
        < result["ema_50"]
        < result["ema_200"]
    ):
        trend = "Strong bearish trend"

    elif result["ema_20"] > result["ema_50"]:
        trend = "Short-term bullish trend"

    else:
        trend = "Mixed trend"

    print("=" * 66)
    print("SWINGPULSE — LIVE STOCK ANALYSIS")
    print("=" * 66)

    print(f"Symbol:                {result['symbol']}")
    print(f"Market data date:      {result['date']}")
    print(f"Latest close:          ${result['close']:.2f}")

    print("\nMODEL OUTPUT")
    print("-" * 66)

    print(
        f"Probability of +{TARGET_RETURN * 100:.0f}% "
        f"within {PREDICTION_HORIZON} trading days: "
        f"{result['probability'] * 100:.2f}%"
    )

    print(
        f"Decision threshold:    "
        f"{DECISION_THRESHOLD * 100:.0f}%"
    )

    print(f"Signal:                {result['signal']}")

    print("\nTECHNICAL INDICATORS")
    print("-" * 66)

    print(
        f"RSI (14):              "
        f"{result['rsi']:.2f} ({rsi_status})"
    )

    print(f"MACD:                  {result['macd']:.4f}")
    print(
        f"MACD Signal:           "
        f"{result['macd_signal']:.4f}"
    )
    print(f"MACD status:           {macd_status}")
    print(f"Trend:                 {trend}")
    print(f"ATR (14):              {result['atr']:.4f}")
    print(
        f"Volatility (20):       "
        f"{result['volatility']:.2%}"
    )

    print("\nDISCLAIMER")
    print("-" * 66)
    print(
        "Experimental model based on historical market data. "
        "Not financial advice."
    )

In [5]:
display_live_analysis("AAPL")

SWINGPULSE — LIVE STOCK ANALYSIS
Symbol:                AAPL
Market data date:      2026-07-10 00:00:00
Latest close:          $315.32

MODEL OUTPUT
------------------------------------------------------------------
Probability of +10% within 10 trading days: 27.03%
Decision threshold:    45%
Signal:                Low Potential

TECHNICAL INDICATORS
------------------------------------------------------------------
RSI (14):              62.93 (Positive momentum)
MACD:                  4.6148
MACD Signal:           1.8940
MACD status:           Bullish
Trend:                 Strong bullish trend
ATR (14):              8.9464
Volatility (20):       2.21%

DISCLAIMER
------------------------------------------------------------------
Experimental model based on historical market data. Not financial advice.


In [6]:
demo_symbols = [
    "AAPL",
    "NVDA",
    "MSFT"
]

demo_results = []

for symbol in demo_symbols:
    result = analyze_live_stock(symbol)

    if result["success"]:
        demo_results.append({
            "Symbol": result["symbol"],
            "Date": result["date"],
            "Close": result["close"],
            "Probability": result["probability"],
            "Signal": result["signal"],
            "RSI_14": result["rsi"],
            "MACD": result["macd"],
            "Volatility_20": result["volatility"]
        })

demo_results_df = pd.DataFrame(
    demo_results
)

demo_results_df["Probability_Percent"] = (
    demo_results_df["Probability"]
    .mul(100)
    .round(2)
)

display(
    demo_results_df[
        [
            "Symbol",
            "Date",
            "Close",
            "Probability_Percent",
            "Signal",
            "RSI_14",
            "MACD",
            "Volatility_20"
        ]
    ].style.format({
        "Close": "${:.2f}",
        "Probability_Percent": "{:.2f}%",
        "RSI_14": "{:.2f}",
        "MACD": "{:.4f}",
        "Volatility_20": "{:.2%}"
    })
)

,Symbol,Date,Close,Probability_Percent,Signal,RSI_14,MACD,Volatility_20
0,AAPL,2026-07-10 00:00:00,$315.32,27.03%,Low Potential,62.93,4.6148,2.21%
1,NVDA,2026-07-10 00:00:00,$210.96,45.98%,Watch,56.97,-1.7351,2.27%
2,MSFT,2026-07-10 00:00:00,$385.10,39.36%,Neutral,47.52,-5.5505,2.36%


In [8]:
user_symbol = input(
    "Enter a stock symbol: "
)

display_live_analysis(user_symbol)

Enter a stock symbol: aapl
SWINGPULSE — LIVE STOCK ANALYSIS
Symbol:                AAPL
Market data date:      2026-07-10 00:00:00
Latest close:          $315.32

MODEL OUTPUT
------------------------------------------------------------------
Probability of +10% within 10 trading days: 27.03%
Decision threshold:    45%
Signal:                Low Potential

TECHNICAL INDICATORS
------------------------------------------------------------------
RSI (14):              62.93 (Positive momentum)
MACD:                  4.6148
MACD Signal:           1.8940
MACD status:           Bullish
Trend:                 Strong bullish trend
ATR (14):              8.9464
Volatility (20):       2.21%

DISCLAIMER
------------------------------------------------------------------
Experimental model based on historical market data. Not financial advice.


In [11]:
demo_scan_symbols = [
    "ORCL",
    "MSFT",
    "NVDA",
    "AMD",
    "META",
    "AMZN",
    "TSLA",
    "GOOGL"
]

rsi_scan_results = []

for symbol in demo_scan_symbols:
    result = analyze_live_stock(symbol)

    if (
        result["success"]
        and 26 <= result["rsi"] <= 35
    ):
        rsi_scan_results.append({
            "Symbol": result["symbol"],
            "Close": result["close"],
            "RSI_14": result["rsi"],
            "Probability": result["probability"],
            "Signal": result["signal"]
        })

rsi_scan_df = pd.DataFrame(
    rsi_scan_results
)

if rsi_scan_df.empty:
    print(
        "No stocks in the demo list currently have "
        "RSI between 26 and 35."
    )
else:
    rsi_scan_df["Probability_Percent"] = (
        rsi_scan_df["Probability"]
        .mul(100)
        .round(2)
    )

    display(
        rsi_scan_df[
            [
                "Symbol",
                "Close",
                "RSI_14",
                "Probability_Percent",
                "Signal"
            ]
        ].style.format({
            "Close": "${:.2f}",
            "RSI_14": "{:.2f}",
            "Probability_Percent": "{:.2f}%"
        })
    )

,Symbol,Close,RSI_14,Probability_Percent,Signal
0,ORCL,$140.64,30.86,47.72%,Watch


In [10]:
print("=" * 66)
print("SWINGPULSE PROJECT SUMMARY")
print("=" * 66)

print("Historical dataset:  3,878,687 feature rows")
print("Stocks processed:    503")
print("Primary model:       Random Forest")
print("Prediction target:   +10% within 10 trading days")
print("Decision threshold:  0.45")
print("Live data source:    Yahoo Finance")
print("Technical features:  19")
print("Agent output:        Probability + technical explanation")
print("Validation:          Completed")

SWINGPULSE PROJECT SUMMARY
Historical dataset:  3,878,687 feature rows
Stocks processed:    503
Primary model:       Random Forest
Prediction target:   +10% within 10 trading days
Decision threshold:  0.45
Live data source:    Yahoo Finance
Technical features:  19
Agent output:        Probability + technical explanation
Validation:          Completed
